# Leave-One-Out Attribution for Next-Token Prediction

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

# Input Definition and Tokenization Analysis

In [2]:
tokenizer.tokenize("The capital of France is")

['The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis']

The leading space is important for the tokenizer.

In [3]:
tokenizer.tokenize("Paris")


['Paris']

In [4]:
tokenizer.tokenize(" Paris")

['ĠParis']

In [5]:
prompt="The capital of France is"
target=" Paris"

 # Next-Token Probability Extraction

In [6]:
tokens=tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
tokens = {k: v.to(device) for k, v in tokens.items()} #for mantaining them in GPU (cuda)

with torch.no_grad():
    outputs = model(**tokens)

print(outputs.logits.shape)


torch.Size([1, 5, 151936])


In [7]:
next_token_logits=outputs.logits[0, -1, :]

In [8]:
import torch.nn.functional as F

#-1 just for general notation
next_token_logprobs = F.log_softmax(next_token_logits.float(), dim=-1)



# Probability Computation

In [9]:
target_id = tokenizer(target, add_special_tokens=False)["input_ids"][0]

#item() for printing clean (not with tensor format)
target_logprob = next_token_logprobs[target_id].item() #log(p)
target_prob = next_token_logprobs[target_id].exp().item() #e^log(p)=p

print("target_id:", target_id)
print("logprob:", target_logprob)
print("prob:", target_prob)

target_id: 12095
logprob: -1.1559944152832031
prob: 0.31474438309669495


In [10]:
print(next_token_logprobs)

tensor([-10.3435, -12.9685, -13.7810,  ..., -24.5310, -24.5310, -24.5310])


# Leave-One-Out Token Importance Analysis

In [11]:
base_logprob = target_logprob

prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
print("Prompt ids:", prompt_ids)

diff = torch.zeros(len(prompt_ids))

for i in range(len(prompt_ids)):
    loo_ids = prompt_ids[:i] + prompt_ids[i+1:]
    loo_tensor = torch.tensor([loo_ids], device=device)

    tokens = {
        "input_ids": loo_tensor,
        "attention_mask": torch.ones_like(loo_tensor)
    }

    with torch.no_grad():
        outputs = model(**tokens)

    next_token_logits = outputs.logits[0, -1, :]
    next_token_logprobs = F.log_softmax(next_token_logits.float(), dim=-1)
    loo_logprob = next_token_logprobs[target_id].item() #better to work in logprobs for numerical stability?
    diff[i] = base_logprob - loo_logprob

print("LOO diffs:", diff)

Prompt ids: [785, 6722, 315, 9625, 374]
LOO diffs: tensor([1.2115, 3.4625, 6.6405, 7.3229, 4.2478])


# Normalization of Token Importance Scores

In [12]:
scores = diff.clone()
total_importance = scores.abs().sum()

if total_importance == 0:
    normalized_scores = torch.zeros_like(scores)
else:
    normalized_scores = scores / total_importance

print("Normalized scores:", normalized_scores)

Normalized scores: tensor([0.0529, 0.1513, 0.2902, 0.3200, 0.1856])


# Token Importance Visualization

In [13]:
from IPython.display import HTML, display

tokens_str = tokenizer.convert_ids_to_tokens(prompt_ids)
print("Tokens list:", tokens_str)
scores_list = normalized_scores.tolist()

Tokens list: ['The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis']


In [14]:
def score_To_Blue(score):
    intensity = int(255 * (1 - score))
    return f"#{intensity:02x}{intensity:02x}ff"

colored_tokens = []
for tok, score in zip(tokens_str, scores_list):
    tok = tok.replace("Ġ", " ")
    color = score_To_Blue(score)
    html_token = f'<span style="background-color: {color}; color: white; padding: 2px; margin: 1px;">{tok}</span>'
    colored_tokens.append(html_token)

In [15]:
html_text = "".join(colored_tokens)
display(HTML(html_text))